# ELItis — Tier-list Thumbnail Generator

**Workflow**

| Cell | Purpose | Re-run? |
|------|---------|---------|
| **0 · Setup** | Install package, set project name and paths | Once per session |
| **1 · Import** | Load CSV labels (additive — re-run to add more CSVs) | Every new CSV |
| **2 · Images** | Match or upload images for unmatched items | When images are missing |
| **3 · Tune** | Adjust visual settings | Until it looks right |
| **4 · Render** | Full-res render → warnings → gallery preview | After every tune |
| **5 · Export** | Download all thumbnails as a zip | Once at the end |

**Tips**
- Labels come from CSV; images are matched by filename stem or assigned manually in Cell 2
- If you already have a project `.json` from the desktop app, drop it in the zip — settings and image paths are loaded automatically and Cell 3 can be skipped
- Items with no image render as a grey checkerboard and are flagged 🔴 in the warnings table
- Loop between **3 → 4** until the gallery looks right, then run **5** once

In [ ]:
# ── Cell 1 · Setup & Data Loader ─────────────────────────────────────────────
# Run once per session.
#
# Upload a zip containing any combination of:
#   *.json          — project file saved by the ELItis desktop app (recommended)
#   *.png/jpg/…     — background images
#   *.csv / *.txt   — label list (used only when no project JSON is present)
#   Fonts/          — font folder (optional; falls back to PIL built-in)
#
# Zip layout can be flat or nested — the loader finds everything by extension.
# ─────────────────────────────────────────────────────────────────────────────

REPO_URL = "https://github.com/JonnyDanny/ELitis"

import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", f"git+{REPO_URL}", "-q"], check=True)

import io as _io, json, shutil, zipfile
from pathlib import Path

from google.colab import files
from IPython.display import display, HTML

PROJECT_DIR = Path("/content/project")
OUTPUT_DIR  = Path("/content/output")

if PROJECT_DIR.exists():
    shutil.rmtree(PROJECT_DIR)
PROJECT_DIR.mkdir(parents=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# ── Upload & extract ──────────────────────────────────────────────────────────
print("Upload your project zip:")
uploaded = files.upload()
if not uploaded:
    raise RuntimeError("No file uploaded.")

with zipfile.ZipFile(_io.BytesIO(list(uploaded.values())[0])) as z:
    z.extractall(PROJECT_DIR)
print(f"Extracted to {PROJECT_DIR}")

# ── Discover contents ─────────────────────────────────────────────────────────
IMAGE_EXTS = {".png", ".jpg", ".jpeg", ".webp", ".bmp", ".tiff", ".tif"}
LABEL_EXTS = {".csv", ".tsv", ".txt"}

all_images = sorted(p for p in PROJECT_DIR.rglob("*") if p.suffix.lower() in IMAGE_EXTS)
all_jsons  = sorted(p for p in PROJECT_DIR.rglob("*.json"))
all_labels = sorted(p for p in PROJECT_DIR.rglob("*") if p.suffix.lower() in LABEL_EXTS)

# Lookup table: filename (lower) → Path, and stem (lower) → Path
_img_by_name: dict[str, Path] = {}
for p in all_images:
    _img_by_name[p.name.lower()] = p
    _img_by_name[p.stem.lower()] = p

def _remap(path_str: str | None) -> str | None:
    """Translate a Windows-absolute image path to the extracted Colab path."""
    if not path_str:
        return None
    p = Path(path_str)
    match = _img_by_name.get(p.name.lower()) or _img_by_name.get(p.stem.lower())
    return str(match) if match else path_str

# ── Load or build project ─────────────────────────────────────────────────────
from elitis.core import data_io
from elitis.core.models import Project

if all_jsons:
    project = data_io.load_project(all_jsons[0])
    print(f"Loaded project '{project.name}' — {len(project.content_items)} items")
    for item in project.items:
        item.image_path = _remap(item.image_path)
else:
    # Build from labels + auto-match images by stem
    if all_labels:
        labels = data_io.import_labels(all_labels[0])
        print(f"Imported {len(labels)} labels from {all_labels[0].name}")
    else:
        labels = [p.stem for p in all_images]
        print(f"No label file found — using {len(labels)} image filenames as labels")

    project = Project.new("imported", str(OUTPUT_DIR))
    for label in labels:
        item = project.add_item(label)
        item.image_path = _img_by_name.get(label.lower().replace(" ", "_")) or \
                          _img_by_name.get(label.lower())
        if item.image_path:
            item.image_path = str(item.image_path)
    print(f"Created project '{project.name}' — {len(project.content_items)} items")

# ── Summary table ─────────────────────────────────────────────────────────────
rows = "".join(
    f"<tr><td style='padding:2px 16px 2px 0'>{item.label or '—'}</td>"
    f"<td style='padding:2px 0'>{Path(item.image_path).name if item.image_path else '⚠ no image'}</td></tr>"
    for item in project.content_items
)
display(HTML(
    "<b>Items ready to render:</b><br>"
    "<div style='max-height:300px;overflow-y:auto'>"
    "<table style='border-collapse:collapse;font-size:13px'>"
    "<tr><th style='text-align:left;padding:2px 16px 2px 0'>Label</th>"
    "<th style='text-align:left'>Image</th></tr>"
    + rows + "</table></div>"
))

In [ ]:
# ── Cell 2 · Tuner ───────────────────────────────────────────────────────────
# Adjust visual settings for the project.
#
# Skip this cell entirely if you loaded a project.json from the desktop app —
# all settings are already baked into the project object.
#
# Full ipywidgets UI — coming soon.
# For now, override individual settings programmatically:
#
#   project.defaults.settings.font_size.value = 72
#   project.defaults.settings.font_size.use_default = False
#
#   project.defaults.settings.box_enabled.value = False
#   project.defaults.settings.box_enabled.use_default = False
#
# See elitis/core/models.py → FIELD_DEFAULTS for all available field names.
# ─────────────────────────────────────────────────────────────────────────────

print("Tuner — no changes applied. Settings from Cell 1 are in effect.")
print("Edit this cell to override settings, then re-run Cell 3.")

In [ ]:
# ── Cell 3 · Render & Gallery ─────────────────────────────────────────────────
# Renders all items at full resolution, then shows a scrollable gallery.
# Cell 4 just zips what this cell wrote — no re-render needed.
# ─────────────────────────────────────────────────────────────────────────────

import base64, io as _io, shutil
from pathlib import Path
from PIL import Image
from IPython.display import display, HTML

from elitis.core import renderer
from elitis.core.font_manager import FontManager

FONTS_DIR  = Path("/content/project/Fonts")
OUTPUT_DIR = Path("/content/output")

font_manager = FontManager(FONTS_DIR if FONTS_DIR.exists() else Path("/nonexistent"))

if OUTPUT_DIR.exists():
    shutil.rmtree(OUTPUT_DIR)
OUTPUT_DIR.mkdir(parents=True)

items = project.content_items
print(f"Rendering {len(items)} items...")

saved = renderer.render_all(
    project, font_manager, OUTPUT_DIR,
    on_progress=lambda d, t, p: print(f"  [{d}/{t}] {p.name}"),
)
print(f"\n✓ {len(saved)} images written to {OUTPUT_DIR}")

# ── Inline gallery ────────────────────────────────────────────────────────────
THUMB = (320, 180)

def _thumb_b64(path: Path) -> str:
    img = Image.open(path)
    img.thumbnail(THUMB, Image.Resampling.LANCZOS)
    buf = _io.BytesIO()
    img.save(buf, format="PNG")
    return base64.b64encode(buf.getvalue()).decode()

cards = "".join(
    f"<div style='display:inline-block;margin:6px;text-align:center;vertical-align:top'>"
    f"<img src='data:image/png;base64,{_thumb_b64(p)}' style='border-radius:4px;display:block'/>"
    f"<small style='color:#ccc'>{p.stem}</small></div>"
    for p in sorted(OUTPUT_DIR.glob("*"))
)

display(HTML(
    "<div style='max-height:640px;overflow-y:auto;background:#1a1a1a;"
    "padding:10px;border-radius:6px;line-height:1.2'>"
    + cards + "</div>"
))

In [ ]:
# ── Cell 4 · Export ───────────────────────────────────────────────────────────
# Zips everything Cell 3 rendered and starts a download.
# Run Cell 3 first. No re-render here.
# ─────────────────────────────────────────────────────────────────────────────

import io as _io, zipfile
from pathlib import Path
from google.colab import files

OUTPUT_DIR = Path("/content/output")
images = sorted(OUTPUT_DIR.glob("*"))
if not images:
    raise RuntimeError("No rendered images found — run Cell 3 first.")

zip_name = f"{project.name}_thumbnails.zip"
zip_path = Path(f"/content/{zip_name}")

with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as zf:
    for p in images:
        zf.write(p, p.name)

print(f"Downloading {len(images)} images as '{zip_name}' ...")
files.download(str(zip_path))